In [13]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from pathlib import Path


# 9 Feature Engineering

Feature engineering transforms the cleaned datasets into a model-ready feature matrix by creating informative variables from historical sales, calendar information, store metadata, promotions, holidays, and external factors.

The objective is to capture temporal patterns, seasonality, business events, and historical demand signals that improve forecasting accuracy.

## Workflow

The feature engineering process consists of:

1. Preparing a unified dataset.
2. Creating calendar-based features.
3. Encoding cyclical time patterns.
4. Generating lag features.
5. Computing rolling statistics.
6. Incorporating external variables.
7. Building the final feature matrix for modeling.

In [2]:
PROCESSED_DATA = Path("../data/processed")

train = pd.read_csv(PROCESSED_DATA / "train.csv", parse_dates=["date"])
oil = pd.read_csv(PROCESSED_DATA / "oil.csv", parse_dates=["date"])
transactions = pd.read_csv(PROCESSED_DATA / "transactions.csv", parse_dates=["date"])

stores = pd.read_csv("../data/raw/stores.csv")
holidays = pd.read_csv("../data/raw/holidays_events.csv", parse_dates=["date"])
test = pd.read_csv("../data/raw/test.csv", parse_dates=["date"])

# 10. Prepare Feature Engineering Dataset

The training and testing datasets are combined before feature engineering so that identical transformations are applied consistently.

The combined dataset is sorted chronologically for each store and product family, ensuring that lag and rolling-window calculations are computed in the correct temporal order.

In [3]:
# Merge train and test to compute features consistently
df = pd.concat([train, test], ignore_index=True, sort=False)
df = df.sort_values(["store_nbr", "family", "date"]).reset_index(drop=True)


# 10.1 Calendar Features

Calendar features capture temporal patterns directly from the date column.

These variables help the model identify recurring trends such as weekday effects, monthly seasonality, weekends, and the progression of time throughout the dataset. They provide a strong baseline representation of temporal behavior before more advanced lag and rolling features are introduced.

In [4]:
# Calendar features
df["day_of_week"]  = df["date"].dt.dayofweek
df["month"]        = df["date"].dt.month
df["year"]         = df["date"].dt.year
df["is_weekend"]   = (df["date"].dt.dayofweek >= 5).astype(int)
df["day_of_year"]  = df["date"].dt.dayofyear
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["date_index"]   = (df["date"] - df["date"].min()).dt.days

# 10.2 Cyclical (Fourier) Features

Many calendar variables are cyclical rather than linear. For example, Sunday is followed by Monday, and December is followed by January.

To preserve this cyclical nature, sine and cosine transformations are applied to weekly and yearly time components. These Fourier features allow the model to learn smooth seasonal patterns without introducing artificial discontinuities at cycle boundaries.

In [5]:
# Fourier features for smooth annual and weekly seasonality
df["sin_day"]  = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["cos_day"]  = np.cos(2 * np.pi * df["day_of_year"] / 365)
df["sin_week"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["cos_week"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

# 10.3 Lag Features

Lag features capture historical sales from previous time periods and are among the most informative predictors in time series forecasting.

Instead of relying only on the current date, the model can learn how recent demand, weekly patterns, monthly trends, and yearly seasonality influence future sales.

The lag intervals used in this project represent both short-term memory (1–7 days), medium-term behavior (2–8 weeks), and annual seasonality (364–365 days).

In [6]:
# Lag features
# Lags 1-6: short-term dynamics
# Lags 7/14/21/28/42/56: weekly and monthly cycles
# Lags 364/365: sales approximately one year ago
for lag in [1, 2, 3, 4, 5, 6, 7, 14, 21, 28, 42, 56, 364, 365]:
    df[f"lag_{lag}"] = (
        df.groupby(["store_nbr", "family"])["sales"]
        .shift(lag)
    )

# 10.4 Rolling Statistics

While lag features capture sales on specific previous days, rolling statistics summarize demand over a recent time window.

Rolling averages reduce daily fluctuations and provide a smoother representation of demand trends. They help the model distinguish between temporary spikes and sustained changes in sales patterns.

The rolling windows used in this project capture short-term, medium-term, and annual demand behavior.

In [7]:
# Rolling mean: average of past N days, excluding current day
for window in [7, 14, 28]:
    df[f"rolling_mean_{window}"] = (
        df.groupby(["store_nbr", "family"])["sales"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

# Annual rolling mean
df["rolling_mean_364"] = (
    df.groupby(["store_nbr", "family"])["sales"]
    .transform(lambda x: x.shift(1).rolling(364, min_periods=30).mean())
)

# 10.5 Oil Price Features

Oil prices are included as an external economic indicator that may influence consumer spending and purchasing behavior.

In addition to the daily oil price, rolling averages are computed to capture longer-term trends while reducing short-term volatility.

In [8]:
# Add oil price
df = df.merge(oil[["date", "dcoilwtico"]], on="date", how="left")

# 10.6 Store Features

Store metadata provides static characteristics of each retail location, such as store type and cluster.

These features allow the model to learn systematic differences in sales patterns across different stores.

In [9]:
# Add store type and cluster
df = df.merge(
    stores[["store_nbr", "type", "cluster"]],
    on="store_nbr", how="left"
).rename(columns={"type": "store_type"})


# 10.7 Promotion Features

Promotional campaigns are one of the strongest drivers of retail sales.

Along with the daily promotion count, rolling averages are computed to capture sustained promotional activity, enabling the model to distinguish between short-term promotions and longer campaigns.

In [10]:
# Add transaction lags 16-23 days back
# For test, these lags always point to the train period
# Example: 2017-08-16 minus 16 days = 2017-07-31
df = df.merge(
    transactions[["date", "store_nbr", "transactions"]],
    on=["date", "store_nbr"], how="left"
)
for lag in range(16, 24):
    df[f"transactions_lag_{lag}"] = df.groupby("store_nbr")["transactions"].shift(lag)
df = df.drop(columns=["transactions"])

# Rolling means for oil and promotions
for col in ["dcoilwtico", "onpromotion"]:
    for w in [7, 28]:
        df[f"{col}_ma{w}"] = (
            df.groupby(["store_nbr", "family"])[col]
            .transform(lambda x: x.rolling(w, min_periods=1).mean())
        )


# 10.8 Holiday Features

Public holidays and major commercial events can substantially influence retail demand.

This project incorporates national holiday indicators along with features representing the day before a holiday and major shopping events such as Black Friday and Cyber Monday. These variables help the model capture demand shifts associated with holidays and special retail occasions.

In [11]:
# National holidays excluding transferred dates
national_hol = holidays[
    (holidays["locale"] == "National") &
    (holidays["transferred"] == False)
]["date"].unique()

df["is_holiday_national"] = df["date"].isin(national_hol).astype(int)
df["day_before_holiday"]  = (df["date"] + pd.Timedelta(days=1)).isin(national_hol).astype(int)

# Commercial events
df["is_black_friday"] = df["date"].isin(
    holidays[holidays["description"] == "Black Friday"]["date"].unique()
).astype(int)

df["is_cyber_monday"] = df["date"].isin(
    holidays[holidays["description"] == "Cyber Monday"]["date"].unique()
).astype(int)

# Terremoto Manabi period
df["is_terremoto"] = df["date"].isin(
    holidays[holidays["description"].str.contains("Terremoto", na=False)]["date"].unique()
).astype(int)

# Individual national holidays
_hol = holidays[(holidays["locale"] == "National") & (holidays["transferred"] == False)]
df["is_navidad"]       = df["date"].isin(_hol[_hol["description"].str.contains("Navidad",      na=False)]["date"].unique()).astype(int)
df["is_dia_madre"]     = df["date"].isin(_hol[_hol["description"].str.contains("Madre",        na=False)]["date"].unique()).astype(int)
df["is_futbol"]        = df["date"].isin(_hol[_hol["description"].str.contains("futbol|F\xfatbol|Mundial", na=False)]["date"].unique()).astype(int)
df["is_dia_trabajo"]   = df["date"].isin(_hol[_hol["description"].str.contains("Trabajo",      na=False)]["date"].unique()).astype(int)
df["is_primer_dia"]    = df["date"].isin(_hol[_hol["description"].str.contains("Primer",       na=False)]["date"].unique()).astype(int)
df["is_dia_difuntos"]  = df["date"].isin(_hol[_hol["description"].str.contains("Difuntos",     na=False)]["date"].unique()).astype(int)

# Working days marked in the holidays table
work_day_dates = holidays[
    (holidays["type"] == "Work Day") |
    (holidays["description"].str.contains("trabajo|Work", case=False, na=False) & (holidays["locale"] == "National"))
]["date"].unique()
df["work_day"] = df["date"].isin(work_day_dates).astype(int)


# 10.9 Final Feature Matrix

After all engineered features have been created, the combined dataset is separated back into the training and testing sets.

The resulting feature matrix contains historical demand signals, calendar variables, cyclical encodings, rolling statistics, promotions, store characteristics, holidays, and external economic indicators, providing the input for the forecasting models developed in the next notebook.

In [14]:
# Split data back into train and test
train_fe = df[df["sales"].notna()].copy()
test_fe  = df[df["sales"].isna()].copy()

train_fe = train_fe.drop(columns=["id", "day_name"], errors="ignore")
test_fe  = test_fe.drop(columns=["id", "day_name"], errors="ignore")

print("train_fe shape:", train_fe.shape)
print("test_fe shape:", test_fe.shape)
print()
print("Features:", train_fe.columns.tolist())
print()
print("Missing values in train_fe:")
print(train_fe.isnull().sum()[train_fe.isnull().sum() > 0])

PROCESSED_DATA.mkdir(exist_ok=True)

train_fe.to_csv(PROCESSED_DATA / "train_fe.csv", index=False)
test_fe.to_csv(PROCESSED_DATA / "test_fe.csv", index=False)
df.to_csv(PROCESSED_DATA / "feature_matrix.csv", index=False)


train_fe shape: (3008016, 61)
test_fe shape: (28512, 61)

Features: ['date', 'store_nbr', 'family', 'sales', 'onpromotion', 'day_of_week', 'month', 'year', 'is_weekend', 'day_of_year', 'week_of_year', 'date_index', 'sin_day', 'cos_day', 'sin_week', 'cos_week', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_42', 'lag_56', 'lag_364', 'lag_365', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_mean_364', 'dcoilwtico', 'store_type', 'cluster', 'transactions_lag_16', 'transactions_lag_17', 'transactions_lag_18', 'transactions_lag_19', 'transactions_lag_20', 'transactions_lag_21', 'transactions_lag_22', 'transactions_lag_23', 'dcoilwtico_ma7', 'dcoilwtico_ma28', 'onpromotion_ma7', 'onpromotion_ma28', 'is_holiday_national', 'day_before_holiday', 'is_black_friday', 'is_cyber_monday', 'is_terremoto', 'is_navidad', 'is_dia_madre', 'is_futbol', 'is_dia_trabajo', 'is_primer_dia', 'is_dia_difuntos', 'work_day']

Missing values in tr

# 11. Feature Engineering Summary

This notebook transformed the cleaned datasets into a model-ready feature matrix by engineering informative predictors from multiple data sources.

### Features Created

- Calendar features
- Cyclical (Fourier) features
- Lag features
- Rolling statistics
- Oil price features
- Promotion features
- Store metadata
- Holiday indicators

These engineered features provide the foundation for the machine learning models developed in the next notebook.